In [2]:
import os
import pandas as pd


In [5]:
wb = pd.read_csv("data/cleaned/worldbank_clean.csv").convert_dtypes()
imf = pd.read_csv("data/cleaned/imf_clean.csv").convert_dtypes()
oecd = pd.read_csv("data/cleaned/oecd_clean.csv").convert_dtypes()
print(wb.shape)
print(imf.shape)
print(oecd.shape)

(1018, 4)
(1007, 3)
(48, 6)


In [6]:
wb = wb.rename(columns={"Country Name": "Country"})
wb.head()

,Country,Country Code,Year,TertiaryEnrollment
0,Aruba,ABW,2024,14.065542
1,Afghanistan,AFG,2018,9.99592
2,Afghanistan,AFG,2020,10.85436
3,Angola,AGO,2023,10.049358
4,Angola,AGO,2024,9.954907


In [11]:
merged = pd.merge(wb, imf, how="outer", on=["Country", "Year"], indicator=True)
merged["_merge"].value_counts()

_merge
both          746
left_only     272
right_only    261
Name: count, dtype: int64

In [12]:
merged = pd.merge(merged, oecd, how="left", on=["Country", "Year"])
merged[merged["Year"] == 2023].head()

,Country,Country Code,Year,TertiaryEnrollment,UnemploymentRate,_merge,qc_flag,attain_below_upper_sec,attain_upper_sec,attain_tertiary
8,Albania,ALB,2023,64.729352,9.5,both,<NA>,<NA>,<NA>,<NA>
17,Algeria,DZA,2023,55.532019,<NA>,left_only,<NA>,<NA>,<NA>,<NA>
25,Andorra,AND,2023,62.410694,1.525,both,<NA>,<NA>,<NA>,<NA>
28,Angola,AGO,2023,10.049358,<NA>,left_only,<NA>,<NA>,<NA>,<NA>
36,Argentina,ARG,2023,107.822574,6.125,both,ok,32.177174,43.569042,23.67523


In [13]:
matched = merged[merged["_merge"] == "both"]
unmatched = merged[merged["_merge"] != "both"]

In [20]:
oecd_unmatched = oecd[~oecd["Country"].isin(matched[matched["Year"] == 2023]["Country"])]

print("Matched records:", len(matched))
print("Unmatched records:", len(unmatched))
print("OECD countries not linked:", oecd_unmatched["Country"].tolist())

Matched records: 746
Unmatched records: 533
OECD countries not linked: ['Brazil', 'Colombia', 'Costa Rica', 'Ireland', 'Peru', 'United States']


In [21]:
os.makedirs("data/integrated", exist_ok=True)

merged.to_csv("data/integrated/integrated_full.csv", index=False)
matched.to_csv("data/integrated/integrated_matched.csv", index=False)
unmatched.to_csv("data/integrated/unmatched.csv", index=False)